## 01: Data Exploration

Loads ogbg-molhiv through OGB's PygGraphPropPredDataset loader using the official scaffold split. Reports the class imbalance ratio and graph size statistics found in the data, and documents what OGB's atom/bond featurization actually encodes. Builds PyG DataLoader objects from the official split indices for the training notebooks.

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = "/content/drive/MyDrive/molhiv-gnn"

# !pip install torch-geometric ogb rdkit

BASE_DIR = ".."

In [2]:
import pandas as pd
import torch
import numpy as np
import torch_geometric
from torch_geometric.loader import DataLoader
from ogb.graphproppred import PygGraphPropPredDataset

torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataEdgeAttr,
    torch_geometric.data.data.DataTensorAttr,
    torch_geometric.data.storage.GlobalStorage,
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
dataset = PygGraphPropPredDataset(name="ogbg-molhiv", root=f"{BASE_DIR}/dataset")
split_idx = dataset.get_idx_split()
len(dataset), {k: len(v) for k, v in split_idx.items()}

(41127, {'train': 32901, 'valid': 4113, 'test': 4113})

In [4]:
for split_name, idx in split_idx.items():
    labels = torch.cat([dataset[i].y for i in idx]).squeeze()
    n_pos = int((labels == 1).sum())
    n_neg = int((labels == 0).sum())
    print(f"{split_name:>6}: total={len(idx):>6}  pos={n_pos:>5}  neg={n_neg:>6}  pos_ratio={n_pos / len(idx):.4f}")

 train: total= 32901  pos= 1232  neg= 31669  pos_ratio=0.0374


 valid: total=  4113  pos=   81  neg=  4032  pos_ratio=0.0197


  test: total=  4113  pos=  130  neg=  3983  pos_ratio=0.0316


In [5]:
num_nodes = np.array([data.num_nodes for data in dataset])
num_edges = np.array([data.num_edges for data in dataset])

print("nodes per graph -> mean: {:.2f}  min: {}  max: {}".format(num_nodes.mean(), num_nodes.min(), num_nodes.max()))
print("edges per graph -> mean: {:.2f}  min: {}  max: {}".format(num_edges.mean(), num_edges.min(), num_edges.max()))

nodes per graph -> mean: 25.51  min: 2  max: 222
edges per graph -> mean: 54.94  min: 2  max: 502


In [6]:
from ogb.utils.features import get_atom_feature_dims, get_bond_feature_dims

sample = dataset[0]
print("x shape:", sample.x.shape)
print("edge_attr shape:", sample.edge_attr.shape)
print("atom feature dims (num categories per column):", get_atom_feature_dims())
print("bond feature dims (num categories per column):", get_bond_feature_dims())
print(sample.x[:5])
print(sample.edge_attr[:5])

x shape: torch.Size([19, 9])
edge_attr shape: torch.Size([40, 3])
atom feature dims (num categories per column): [119, 5, 12, 12, 10, 6, 6, 2, 2]
bond feature dims (num categories per column): [5, 6, 2]
tensor([[ 5,  0,  4,  5,  3,  0,  2,  0,  0],
        [ 5,  0,  4,  5,  2,  0,  2,  0,  0],
        [ 5,  0,  3,  5,  0,  0,  1,  0,  1],
        [ 7,  0,  2,  6,  0,  0,  1,  0,  1],
        [28,  0,  4,  2,  0,  0,  5,  0,  1]])
tensor([[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [1, 0, 0]])


In [7]:
train_loader = DataLoader(dataset[split_idx["train"]], batch_size=128, shuffle=True)
valid_loader = DataLoader(dataset[split_idx["valid"]], batch_size=256, shuffle=False)
test_loader = DataLoader(dataset[split_idx["test"]], batch_size=256, shuffle=False)

batch = next(iter(train_loader))
batch

DataBatch(edge_index=[2, 7024], edge_attr=[7024, 3], x=[3295, 9], y=[128, 1], num_nodes=3295, batch=[3295], ptr=[129])